In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType,
    TimestampType, LongType
)

audit_table = "myadlasdevcalaog.gold.pipeline_audit_log"

schema = StructType([
    StructField("pipeline_name", StringType(), False),
    StructField("run_status", StringType(), False),
    StructField("source_row_count", LongType(), True),
    StructField("target_row_count", LongType(), True),
    StructField("run_timestamp", TimestampType(), False)
])

empty_df = spark.createDataFrame([], schema)

empty_df.write.format("delta").mode("ignore").saveAsTable(audit_table)

print("Audit table is ready:", audit_table)

Audit table is ready: myadlasdevcalaog.gold.pipeline_audit_log


In [0]:
from pyspark.sql import functions as F

audit_record = [(
    "Customer_Medallion_Pipeline",
    "SUCCESS",
    2,
    6,
    None
)]

audit_df = spark.createDataFrame(
    audit_record,
    ["pipeline_name", "run_status", "source_row_count",
     "target_row_count", "run_timestamp"]
).withColumn("run_timestamp", F.current_timestamp())

audit_df.write.format("delta").mode("append").saveAsTable(audit_table)

display(spark.table(audit_table).orderBy(F.col("run_timestamp").desc()))

---------------------------------------------------------------------------
PySparkValueError                         Traceback (most recent call last)
File <command-8965521408008026>, line 11
      1 from pyspark.sql import functions as F
      3 audit_record = [(
      4     "Customer_Medallion_Pipeline",
      5     "SUCCESS",
   (...)      8     None
      9 )]
---> 11 audit_df = spark.createDataFrame(
     12     audit_record,
     13     ["pipeline_name", "run_status", "source_row_count",
     14      "target_row_count", "run_timestamp"]
     15 ).withColumn("run_timestamp", F.current_timestamp())
     17 audit_df.write.format("delta").mode("append").saveAsTable(audit_table)
     19 display(spark.table(audit_table).orderBy(F.col("run_timestamp").desc()))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:820, in SparkSession.createDataFrame(self, data, schema, samplingRatio, verifySchema)
    815         _num_cols = len(_cols)
    817     if _has

In [0]:

from pyspark.sql import functions as F

audit_table = "myadlasdevcalaog.gold.pipeline_audit_log"

audit_df = spark.createDataFrame(
    [
        ("Customer_Medallion_Pipeline", "SUCCESS", 2, 6)
    ],
    [
        "pipeline_name",
        "run_status",
        "source_row_count",
        "target_row_count"
    ]
).withColumn(
    "run_timestamp",
    F.current_timestamp()
)

audit_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(audit_table)

display(
    spark.table(audit_table)
    .orderBy(F.col("run_timestamp").desc())
)


pipeline_name,run_status,source_row_count,target_row_count,run_timestamp
Customer_Medallion_Pipeline,SUCCESS,2,6,2026-10-22T08:34:12.092Z


In [0]:

from pyspark.sql import functions as F

silver_table = "myadlasdevcalaog.silver.customers"
audit_table = "myadlasdevcalaog.gold.pipeline_audit_log"

# Count actual distinct customers in Silver
target_count = (
    spark.table(silver_table)
    .select("customer_id")
    .distinct()
    .count()
)

# Count the current simulated source records
source_count = df_changes_deduplicated.count()

# Build an audit record using actual counts
audit_record = spark.createDataFrame(
    [(
        "Customer_Medallion_Pipeline",
        "SUCCESS",
        int(source_count),
        int(target_count)
    )],
    ["pipeline_name", "run_status", "source_row_count", "target_row_count"]
).withColumn("run_timestamp", F.current_timestamp())

audit_record.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(audit_table)

display(
    spark.table(audit_table)
    .orderBy(F.col("run_timestamp").desc())
)


In [0]:

from pyspark.sql import functions as F

silver_table = "myadlasdevcalaog.silver.customers"
audit_table = "myadlasdevcalaog.gold.pipeline_audit_log"

# Count actual distinct customers in Silver
target_count = (
    spark.table(silver_table)
    .select("customer_id")
    .distinct()
    .count()
)

# Record the number of incoming change records for this exercise
source_count = 2

# Create an audit record
audit_record = spark.createDataFrame(
    [(
        "Customer_Medallion_Pipeline",
        "SUCCESS",
        source_count,
        target_count
    )],
    [
        "pipeline_name",
        "run_status",
        "source_row_count",
        "target_row_count"
    ]
).withColumn("run_timestamp", F.current_timestamp())

audit_record.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(audit_table)

display(
    spark.table(audit_table)
    .orderBy(F.col("run_timestamp").desc())
)


pipeline_name,run_status,source_row_count,target_row_count,run_timestamp
Customer_Medallion_Pipeline,SUCCESS,2,5,2026-10-22T08:36:47.194Z
Customer_Medallion_Pipeline,SUCCESS,2,6,2026-10-22T08:34:12.092Z


In [0]:
silver_table = "myadlasdevcalaog.silver.customers"

print("Total rows:", spark.table(silver_table).count())

print(
    "Distinct customer IDs:",
    spark.table(silver_table).select("customer_id").distinct().count()
)

display(
    spark.table(silver_table)
    .select("customer_id", "customer_name", "city")
    .orderBy("customer_id")
)

Total rows: 5
Distinct customer IDs: 5


customer_id,customer_name,city
101,Amit,Pune
102,Priya,Mumbai
103,Rahul,Nashik
104,Sneha,Pune
105,Vijay,Nagpur


In [0]:
%sql
DESCRIBE HISTORY myadlasdevcalaog.silver.customers;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1398,2026-10-22T09:23:35.000Z,76584891613541,mayur.rpatil078@gmail.com,MERGE,"Map(predicate -> [""(customer_id#174444L = customer_id#174132L)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""(source_updated_date#174136 >= source_updated_date#174448)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","List(54650851123132, Customer_Medallion_Pipeline, 208453703328476, 541134663546328, 76584891613541, manual, 7474650859800015)",List(1892856252024776),01c374b5-0aa2-4ac8-b98b-4c6d2797a9e1,1022-055905-e0lvjzk-v2n,1397,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2276, numTargetBytesRemoved -> 2277, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 3238, materializeSourceTimeMs -> 426, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1230, numTargetRowsUpdated -> 5, numOutputRows -> 5, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1543)",null,Databricks-Runtime/19.11.x-aarch64-photon-scala2.13
1397,2026-10-22T09:21:34.000Z,76584891613541,mayur.rpatil078@gmail.com,MERGE,"Map(predicate -> [""(customer_id#171814L = customer_id#171502L)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""(source_updated_date#171506 >= source_updated_date#171818)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","List(54650851123132, Customer_Medallion_Pipeline, 821294922806327, 1014951663770389, 76584891613541, manual, 7474650859800015)",List(1892856252024776),174b5035-8056-4393-b534-ce809fa68327,1022-055905-e0lvjzk-v2n,1396,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2277, numTargetBytesRemoved -> 2276, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 3493, materializeSourceTimeMs -> 420, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1225, numTargetRowsUpdated -> 5, numOutputRows -> 5, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1814)",null,Databricks-Runtime/19.11.x-aarch64-photon-scala2.13
1396,2026-10-22T09:19:36.000Z,76584891613541,mayur.rpatil078@gmail.com,MERGE,"Map(predicate -> [""(customer_id#169184L = customer_id#168872L)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""(source_updated_date#168876 >= source_updated_date#169188)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","List(54650851123132, Customer_Medallion_Pipeline, 896418605854642, 468667808130758, 76584891613541, manual, 7474650859800015)",List(1892856252024776),4f4aaff5-2d16-4a80-b640-d6019090f750,1022-055905-e0lvjzk-v2n,1395,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2276, numTargetBytesRemoved -> 2276, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 3247, materializeSourceTimeMs -> 416, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1216, numTargetRowsUpdated

In [0]:
%sql
SELECT customer_id, customer_name, city, source_updated_date
FROM myadlasdevcalaog.silver.customers
ORDER BY customer_id;

customer_id,customer_name,city,source_updated_date
101,Amit,Pune,2026-10-01
102,Priya,Mumbai,2026-10-02
103,Rahul,Nashik,2026-10-03
104,Sneha,Pune,2026-10-04
105,Vijay,Nagpur,2026-10-05


In [0]:
%sql
DESCRIBE HISTORY myadlasdevcalaog.silver.customers LIMIT 15;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1398,2026-10-22T09:23:35.000Z,76584891613541,mayur.rpatil078@gmail.com,MERGE,"Map(predicate -> [""(customer_id#174444L = customer_id#174132L)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""(source_updated_date#174136 >= source_updated_date#174448)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","List(54650851123132, Customer_Medallion_Pipeline, 208453703328476, 541134663546328, 76584891613541, manual, 7474650859800015)",List(1892856252024776),01c374b5-0aa2-4ac8-b98b-4c6d2797a9e1,1022-055905-e0lvjzk-v2n,1397,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2276, numTargetBytesRemoved -> 2277, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 3238, materializeSourceTimeMs -> 426, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1230, numTargetRowsUpdated -> 5, numOutputRows -> 5, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1543)",null,Databricks-Runtime/19.11.x-aarch64-photon-scala2.13
1397,2026-10-22T09:21:34.000Z,76584891613541,mayur.rpatil078@gmail.com,MERGE,"Map(predicate -> [""(customer_id#171814L = customer_id#171502L)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""(source_updated_date#171506 >= source_updated_date#171818)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","List(54650851123132, Customer_Medallion_Pipeline, 821294922806327, 1014951663770389, 76584891613541, manual, 7474650859800015)",List(1892856252024776),174b5035-8056-4393-b534-ce809fa68327,1022-055905-e0lvjzk-v2n,1396,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2277, numTargetBytesRemoved -> 2276, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 3493, materializeSourceTimeMs -> 420, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1225, numTargetRowsUpdated -> 5, numOutputRows -> 5, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1814)",null,Databricks-Runtime/19.11.x-aarch64-photon-scala2.13
1396,2026-10-22T09:19:36.000Z,76584891613541,mayur.rpatil078@gmail.com,MERGE,"Map(predicate -> [""(customer_id#169184L = customer_id#168872L)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""(source_updated_date#168876 >= source_updated_date#169188)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","List(54650851123132, Customer_Medallion_Pipeline, 896418605854642, 468667808130758, 76584891613541, manual, 7474650859800015)",List(1892856252024776),4f4aaff5-2d16-4a80-b640-d6019090f750,1022-055905-e0lvjzk-v2n,1395,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2276, numTargetBytesRemoved -> 2276, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 3247, materializeSourceTimeMs -> 416, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1216, numTargetRowsUpdated

In [0]:
%sql
SELECT
    version,
    timestamp,
    operation,
    operationMetrics
FROM (DESCRIBE HISTORY myadlasdevcalaog.silver.customers)
LIMIT 5;

version,timestamp,operation,operationMetrics
1398,2026-10-22T09:23:35.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2276, numTargetBytesRemoved -> 2277, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 3238, materializeSourceTimeMs -> 426, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1230, numTargetRowsUpdated -> 5, numOutputRows -> 5, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1543)"
1397,2026-10-22T09:21:34.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2277, numTargetBytesRemoved -> 2276, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 3493, materializeSourceTimeMs -> 420, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1225, numTargetRowsUpdated -> 5, numOutputRows -> 5, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1814)"
1396,2026-10-22T09:19:36.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2276, numTargetBytesRemoved -> 2276, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 3247, materializeSourceTimeMs -> 416, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1216, numTargetRowsUpdated -> 5, numOutputRows -> 5, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1568)"
1395,2026-10-22T09:17:36.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2276, numTargetBytesRemoved -> 2275, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 3304, materializeSourceTimeMs -> 434, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1282, numTargetRowsUpdated -> 5, numOutputRows -> 5, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1547)"
1394,2026-10-22T09:15:27.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2275, numTargetBytesRemoved -> 2275, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 3518, materializeSourceTimeMs -> 408, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1138, numTargetRowsUpdated -> 5, numOutputRows -> 5, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1938)"


In [0]:
%sql
SELECT
    version,
    timestamp,
    operation,
    operationMetrics['numSourceRows'],
    operationMetrics['numTargetRowsInserted'],
    operationMetrics['numTargetRowsUpdated'],
    operationMetrics['numTargetRowsDeleted']
FROM (DESCRIBE HISTORY myadlasdevcalaog.silver.customers)
LIMIT 10;

version,timestamp,operation,operationMetrics[numSourceRows],operationMetrics[numTargetRowsInserted],operationMetrics[numTargetRowsUpdated],operationMetrics[numTargetRowsDeleted]
1398,2026-10-22T09:23:35.000Z,MERGE,5,0,5,0
1397,2026-10-22T09:21:34.000Z,MERGE,5,0,5,0
1396,2026-10-22T09:19:36.000Z,MERGE,5,0,5,0
1395,2026-10-22T09:17:36.000Z,MERGE,5,0,5,0
1394,2026-10-22T09:15:27.000Z,MERGE,5,0,5,0
1393,2026-10-22T09:08:36.000Z,MERGE,5,0,5,0
1392,2026-10-22T09:06:34.000Z,MERGE,5,0,5,0
1391,2026-10-22T09:04:34.000Z,MERGE,5,0,5,0
1390,2026-10-22T09:02:37.000Z,MERGE,5,0,5,0
1389,2026-10-22T09:00:32.000Z,MERGE,5,0,5,0


In [0]:
%sql
SELECT customer_id, customer_name, city, source_updated_date
FROM myadlasdevcalaog.silver.customers
WHERE customer_id IN (101, 106)
ORDER BY customer_id;

customer_id,customer_name,city,source_updated_date
101,Amit,Bengaluru,2026-10-14
106,Neha,Pune,2026-10-14


In [0]:
%sql
SELECT city, total_customers, customers_missing_email
FROM myadlasdevcalaog.gold.customer_city_summary
ORDER BY city;

city,total_customers,customers_missing_email
Bengaluru,1,0
Mumbai,1,0
Nagpur,1,0
Nashik,1,1
Pune,2,0


In [0]:
silver_table = "myadlasdevcalaog.silver.customers"
gold_table = "myadlasdevcalaog.gold.customer_city_summary"

silver_count = spark.table(silver_table).count()
gold_city_count = spark.table(gold_table).count()

print("Silver customer rows:", silver_count)
print("Gold city-summary rows:", gold_city_count)

Silver customer rows: 6
Gold city-summary rows: 5


In [0]:

from pyspark.sql import functions as F

silver_table = "myadlasdevcalaog.silver.customers"
gold_table = "myadlasdevcalaog.gold.customer_city_summary"
audit_table = "myadlasdevcalaog.gold.pipeline_audit_log"

# Capture actual table counts
silver_count = spark.table(silver_table).count()
gold_city_count = spark.table(gold_table).count()

# Create a new audit record
audit_record = spark.createDataFrame(
    [(
        "Customer_Medallion_Pipeline",
        "SUCCESS",
        int(silver_count),
        int(gold_city_count)
    )],
    [
        "pipeline_name",
        "run_status",
        "source_row_count",
        "target_row_count"
    ]
).withColumn("run_timestamp", F.current_timestamp())

audit_record.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(audit_table)

display(
    spark.table(audit_table)
    .orderBy(F.col("run_timestamp").desc())
)


pipeline_name,run_status,source_row_count,target_row_count,run_timestamp
Customer_Medallion_Pipeline,SUCCESS,6,5,2026-10-22T10:23:51.637Z
Customer_Medallion_Pipeline,SUCCESS,2,5,2026-10-22T08:36:47.194Z
Customer_Medallion_Pipeline,SUCCESS,2,6,2026-10-22T08:34:12.092Z


In [0]:
from pyspark.sql import functions as F

audit_table = "myadlasdevcalaog.gold.pipeline_audit_log"

display(
    spark.table(audit_table)
    .orderBy(F.col("run_timestamp").desc())
    .limit(1)
)

pipeline_name,run_status,source_row_count,target_row_count,run_timestamp
Customer_Medallion_Pipeline,SUCCESS,6,5,2026-10-22T10:23:51.637Z
